# Anchor baseline: untuned LightGBM

The first submission. Its job is not to score well. Its job is to prove the pipeline
runs end to end and to give every later experiment an honest number to be compared
against.

Deliberately untuned: LightGBM defaults, raw features, no engineering, no early
stopping. The only arguments passed are `random_state` and `verbose`.

**The target-mean baseline the workspace rules normally prescribe is skipped here.**
The metric is AUC, which reads only the ordering of predictions, so a constant scores
exactly 0.5 no matter what constant it is. It carries no information.

Run this with Save & Run All, top to bottom, on a clean kernel. A number produced any
other way does not go in the ledger.

In [1]:
import csv
import time
from datetime import datetime, timezone
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID = "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

RUN_NAME = "lgbm_default_anchor"
RUN_NOTES = "untuned lgbm defaults, raw features, native cat and nan handling"

print("lightgbm", lgb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

lightgbm 4.7.0 | pandas 3.0.5 | numpy 2.5.1


## Paths

Resolved by looking for the data rather than hardcoded, so the same notebook runs from
this repo and from a Kaggle notebook without edits.

In [2]:
def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv locally or under /kaggle/input")


REPO, RAW = locate()
SUB_DIR = REPO / "submissions"
OOF_DIR = REPO / "artifacts" / "oof"
SUB_DIR.mkdir(parents=True, exist_ok=True)
OOF_DIR.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("raw :", RAW)

repo: smartphone-addiction
raw : smartphone-addiction/data/raw


In [3]:
train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
sample = pd.read_csv(RAW / "sample_submission.csv")
print(train.shape, test.shape, sample.shape)
print("target rate:", round(train[TARGET].mean(), 6))

(691369, 14) (296302, 13) (296302, 2)
target rate: 0.709424


## Features

`id` is dropped explicitly. It is a contiguous row index that separates train from test
perfectly, so it is a guaranteed leak if it ever reaches the model. The assertion below
is the check, not the comment.

Missing values are left as NaN and categoricals are left as categories. LightGBM handles
both natively, and imputing at the anchor stage would be an untested modelling decision
smuggled into the baseline.

Category levels are taken from train and test together so the encodings line up. That
uses no target information, so it is not leakage.

In [4]:
FEATURES = [c for c in train.columns if c not in (ID, TARGET)]

for c in CAT_COLS:
    levels = pd.Categorical(pd.concat([train[c], test[c]], ignore_index=True)).categories
    train[c] = pd.Categorical(train[c], categories=levels)
    test[c] = pd.Categorical(test[c], categories=levels)

y = train[TARGET].to_numpy()

# Leak checks. These are assertions rather than prose so they fail loudly.
assert ID not in FEATURES, "id must never be a feature"
assert TARGET not in FEATURES, "target must never be a feature"
assert not (set(train[ID]) & set(test[ID])), "train and test ids overlap"
assert list(FEATURES) == [c for c in test.columns if c != ID], "train/test feature mismatch"

print(len(FEATURES), "features:", FEATURES)

12 features: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact']


## Folds

StratifiedKFold, shuffled, seed 42. The reasoning for this scheme over the alternatives
is written out`; it is not re-argued here.

In [5]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.full(len(train), -1, dtype=int)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i

assert (folds >= 0).all(), "every row must land in exactly one fold"
for i in range(N_SPLITS):
    m = folds == i
    print(f"  fold {i}: n={m.sum():>7,}  pos_rate={y[m].mean():.6f}")

  fold 0: n=138,274  pos_rate=0.709425
  fold 1: n=138,274  pos_rate=0.709425
  fold 2: n=138,274  pos_rate=0.709425
  fold 3: n=138,274  pos_rate=0.709425
  fold 4: n=138,273  pos_rate=0.709423


## Train

One model per fold. Test predictions are averaged across the five fold models, which is
the standard bagging effect and costs nothing.

In [6]:
oof = np.zeros(len(train), dtype=float)
test_pred = np.zeros(len(test), dtype=float)
fold_scores = []
t0 = time.time()

for f in range(N_SPLITS):
    tr_m, va_m = folds != f, folds == f
    model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
    model.fit(train.loc[tr_m, FEATURES], y[tr_m])

    p_va = model.predict_proba(train.loc[va_m, FEATURES])[:, 1]
    oof[va_m] = p_va
    test_pred += model.predict_proba(test[FEATURES])[:, 1] / N_SPLITS

    s = roc_auc_score(y[va_m], p_va)
    fold_scores.append(s)
    print(f"  fold {f}: auc={s:.6f}")

print(f"\ntrained in {time.time() - t0:.1f}s")

  fold 0: auc=0.954049


  fold 1: auc=0.954552


  fold 2: auc=0.955426


  fold 3: auc=0.955882


  fold 4: auc=0.954825

trained in 23.5s


## Score

Both numbers are recorded because AUC does not decompose across folds: the mean of the
per-fold AUCs and the AUC of the pooled out-of-fold vector are genuinely different
quantities. `cv_mean` is the one that goes in the ledger, and comparisons across
experiments must use the same one throughout.

In [7]:
cv_mean = float(np.mean(fold_scores))
cv_std = float(np.std(fold_scores))
pooled = float(roc_auc_score(y, oof))

print(f"CV auc     : {cv_mean:.6f} +/- {cv_std:.6f}")
print(f"pooled OOF : {pooled:.6f}")

if cv_std > 0:
    print(f"\nA gain smaller than {cv_std:.6f} is inside fold noise. Do not believe it "
          f"without a seed sweep.")

CV auc     : 0.954947 +/- 0.000645
pooled OOF : 0.954944

A gain smaller than 0.000645 is inside fold noise. Do not believe it without a seed sweep.


In [8]:
tag = f"{RUN_NAME}_seed{SEED}"
np.save(OOF_DIR / f"{tag}.npy", oof)

sub = sample.copy()
sub[TARGET] = test_pred
sub_path = SUB_DIR / f"{tag}.csv"
sub.to_csv(sub_path, index=False)

assert len(sub) == len(sample), "submission row count changed"
assert sub[TARGET].between(0, 1).all(), "predictions outside [0, 1]"
assert sub[TARGET].nunique() > 1, "constant prediction scores auc 0.5 by definition"
print("wrote", sub_path)
sub.head()

wrote smartphone-addiction/submissions/lgbm_default_anchor_seed42.csv


,id,addicted_label
0,691369,0.996922
1,691370,0.960438
2,691371,0.954775
3,691372,0.968227
4,691373,0.989061


## Ledger

One row per run, appended. Re-running this notebook appends another row rather than
overwriting, which is intended: two runs are two runs. Fill `lb_public` in by hand once
the score comes back from Kaggle.

In [9]:
LEDGER = REPO / "experiments.csv"
COLUMNS = ["id", "utc", "name", "cv_mean", "cv_std", "folds",
           "lb_public", "lb_private", "submitted", "notes"]

rows = []
if LEDGER.exists():
    with LEDGER.open(newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))

exp_id = max((int(r["id"]) for r in rows), default=0) + 1
rows.append({
    "id": str(exp_id),
    "utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "name": RUN_NAME,
    "cv_mean": f"{cv_mean:.6f}",
    "cv_std": f"{cv_std:.6f}",
    "folds": str(N_SPLITS),
    "lb_public": "",
    "lb_private": "",
    "submitted": "no",
    "notes": RUN_NOTES,
})

with LEDGER.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=COLUMNS)
    w.writeheader()
    w.writerows({c: r.get(c, "") for c in COLUMNS} for r in rows)

print(f"logged as experiment {exp_id}")
pd.read_csv(LEDGER)

logged as experiment 1


,id,utc,name,cv_mean,cv_std,folds,lb_public,lb_private,submitted,notes
0,1,2026-08-04 02:15,lgbm_default_anchor,0.954947,0.000645,5,NaN,NaN,no,"untuned lgbm defaults, raw features, native ca..."


## Submit

Run from a terminal in the repo root. The submission does not go through the notebook,
so a failed upload cannot silently look like a success.

```
kaggle competitions submit -c playground-series-s6e8 -f submissions/<file>.csv -m "<name>"
```

Then put the returned public LB score into the `lb_public` column of the row this run
just created, and fill the CV / LB table`. If CV and LB disagree, stop and
diagnose before running anything else.